# Fase 6 — integración y calidad de gold

Cada fila representa una semana objetivo distrital. Se crean dos CSV, uno para `h=2` y otro para `h=4`, desde el silver completo con cobertura de casos. Los primeros rezagos sin historia quedan nulos y se excluyen al formar una matriz de entrenamiento; no se rellenan.

Rosa eligió la etiqueta de **semana distrital elevada**: `casos_Dengue > media histórica + 1,5 desviaciones estándar` de la misma semana epidemiológica en los cinco años anteriores. Para que la etiqueta principal represente un nivel elevado, se exige un mínimo de **2 casos** por recomendación tras analizar su sensibilidad; el mínimo de 1 queda documentado como alternativa para señales locales tempranas. La referencia de 2017–2021 usa el Excel histórico 2012–2016 y trata sus ausencias como cero sin afirmar que fueron ceros notificados. `brote` y `umbral_brote_casos` son etiquetas/metadatos: no entran como predictores.

Las 15 fracciones anuales y la población anual son candidatas reconstruidas; las fracciones 2018–2024 incluyen información de 2025. Las dos marcas `socio_2017_disponible_al_origen` y `socio_fracciones_interpoladas_con_2025` son metadatos de auditoría, no predictores. Las anomalías climáticas y el eje PCA se ajustan solo en el entrenamiento de cada fold y por eso no se precomputan aquí.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import json
import pandas as pd
from src.eda.carga import cargar_integrado
from src.modeling.features import generar_gold, validar_gold
from src.modeling.etiqueta_brote import ConfiguracionBroteEstacional
from src.utils.paths import GOLD, SILVER_INTEGRADO
from src.validation.expectations_integrado import coverage_path, load_case_coverage, validate_complete
panel = cargar_integrado(SILVER_INTEGRADO)
validate_complete(panel, load_case_coverage(coverage_path(SILVER_INTEGRADO)))
print({'filas_silver': len(panel), 'distritos': panel.ubigeo.nunique(), 'llaves_duplicadas': int(panel.duplicated(['ubigeo','anio','semana']).sum())})

{'filas_silver': 30485, 'distritos': 65, 'llaves_duplicadas': 0}


## Generación y lectura de verificación

`generar_gold` comprueba todas las familias antes de escribir y deja un manifiesto con hashes de la fuente, cobertura y salidas. La lectura de vuelta valida los CSV persistidos, no solo los DataFrame en memoria.

In [2]:
REGLA_BROTE = ConfiguracionBroteEstacional(multiplicador=1.5, anios_previos=5, minimo_casos=2)
resumen = generar_gold(SILVER_INTEGRADO, GOLD, brote=REGLA_BROTE)
manifest = json.loads((GOLD / 'manifest_fase6.json').read_text(encoding='utf-8'))
filas = []
etiquetas_por_h = {}
for h in (2, 4):
    archivo = GOLD / resumen[h]['archivo']
    guardado = pd.read_csv(archivo, dtype={'ubigeo': 'string'}, parse_dates=['semana_inicio','origen_inicio','origen_cierre'])
    verificacion = validar_gold(guardado, panel, h, con_brote=True, minimo_casos_brote=REGLA_BROTE.minimo_casos)
    assert verificacion['filas'] == len(panel)
    assert not any(c.endswith(('_x','_y')) for c in guardado)
    etiquetas_por_h[h] = guardado[['ubigeo','anio','semana','brote','umbral_brote_casos']]
    filas.append({'horizonte': h, **resumen[h]})
pd.testing.assert_frame_equal(etiquetas_por_h[2], etiquetas_por_h[4])
display(pd.DataFrame(filas))
print({'manifiesto': str(GOLD / 'manifest_fase6.json'), 'sha256_fuente': manifest['sha256_fuente']})
print({'regla_brote': manifest['regla_brote'], 'positivos_por_anio': manifest['positivos_brote_por_anio']})

,horizonte,filas,distritos,semanas_por_distrito,filas_con_ventanas_completas,filas_sin_ventanas_completas,archivo,sha256,columnas
0,2,30485,65,469,30095,390,meteo_socio_epi_piura_2017_2025_h2_gold.csv,5bb6540b1e7d1c949299dc4e38d32b5d003abdbc1895c2...,48
1,4,30485,65,469,29965,520,meteo_socio_epi_piura_2017_2025_h4_gold.csv,01602d436727909b72a6f7d6e1a3ae5e304cde0444c69e...,48


{'manifiesto': '/Users/kanyewest/Documents/Denguekay/data/gold/manifest_fase6.json', 'sha256_fuente': 'a0c4bb4497eeba0156c8ba581f3a6a64b13c42195606d57c3acc4a8ea0eb7a4e'}
{'regla_brote': {'metodo': 'misma semana epidemiológica de años previos; media + multiplicador * DE muestral; desigualdad estricta', 'multiplicador': 1.5, 'anios_previos': 5, 'minimo_casos': 2, 'fuente_historia': 'epi_piura_semanal.csv', 'sha256_fuente_historia': 'b3955bdd2c0caedf60a3987021a9f6fd67b9f7e63ec855d1db8a15cc4d09023a', 'ausencia_historica': 'sin registro tratado como cero, no cero notificado', 'semana_53': 'se suma a semana 52 para la referencia'}, 'positivos_por_anio': {'2017': 520, '2018': 5, '2019': 0, '2020': 2, '2021': 213, '2022': 354, '2023': 1402, '2024': 503, '2025': 3}}


## Punto de control temporal

La semana objetivo 2025-S52 de Piura (`200101`) tenía un caso en `t−4`; su origen cerró el 2025-11-29. Esta comprobación cruza el gold persistido con el valor ya auditado en la fase 2.

In [3]:
g4 = pd.read_csv(GOLD / resumen[4]['archivo'], dtype={'ubigeo': 'string'}, parse_dates=['semana_inicio','origen_inicio','origen_cierre'])
fila = g4.loc[g4.ubigeo.eq('200101') & g4.anio.eq(2025) & g4.semana.eq(52)].iloc[0]
assert fila.semana_inicio == pd.Timestamp('2025-12-21')
assert fila.origen_cierre == pd.Timestamp('2025-11-29')
assert fila.casos_lag_4 == 1
print(fila[['ubigeo','anio','semana','semana_inicio','origen_inicio','origen_cierre','casos_Dengue','brote','umbral_brote_casos','casos_lag_4','casos_media_4_h4']])

ubigeo                             200101
anio                                 2025
semana                                 52
semana_inicio         2025-12-21 00:00:00
origen_inicio         2025-11-23 00:00:00
origen_cierre         2025-11-29 00:00:00
casos_Dengue                            0
brote                                   0
umbral_brote_casos              12.955214
casos_lag_4                           1.0
casos_media_4_h4                     0.75
Name: 468, dtype: object


## Uso posterior

Se deben definir los folds y las fechas reales de disponibilidad de las fuentes antes de evaluar el modelo. Para cada fold, ajustar climatología y PCA solo con entrenamiento mediante las funciones de `src/modeling/`. Excluir de las variables predictoras `brote`, `umbral_brote_casos`, `casos_Dengue` y los metadatos listados en el manifiesto. `ubigeo` identifica el distrito; no se usa como número continuo. La sensibilidad del mínimo de casos y la interpretación de ceros sin notificación están en `docs/feature_engineering/revision_definicion_brote.md`. La síntesis y el contrato para XGBoost están documentados en `docs/feature_engineering/fase7_sintesis_traspaso.md` y el notebook 23.